# Step D1 — Dataset prep (build manifests + pos_weight.json)

In [1]:

import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path("/kaggle/input")
csv_candidates = list(ROOT.rglob("Data_Entry_2017*.csv"))
assert csv_candidates, "❌ Data_Entry_2017.csv not found under /kaggle/input"
META_CSV = csv_candidates[0]
print("Using metadata:", META_CSV)

# find nested image folders
nested_dirs = [d/"images" for d in META_CSV.parent.iterdir()
               if d.is_dir() and d.name.startswith("images_") and (d/"images").is_dir()]
assert nested_dirs, "❌ No nested image folders found"

# map filenames -> paths
index = {}
for d in nested_dirs:
    for p in d.glob("*.png"):
        index[p.name] = str(p)

CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule',
    'Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema',
    'Fibrosis','Pleural_Thickening','Hernia'
]
ALIASES = {"Pleural Thickening":"Pleural_Thickening","No Finding":"No Finding"}

def encode(lbls):
    if lbls.strip()=="No Finding": return np.zeros(len(CLASSES),dtype=int)
    parts=[ALIASES.get(s.strip(),s.strip()) for s in lbls.split('|')]
    vec=np.zeros(len(CLASSES),dtype=int)
    for i,c in enumerate(CLASSES):
        if c in parts: vec[i]=1
    return vec

df = pd.read_csv(META_CSV)
df["image_path"] = df["Image Index"].map(index.get)
df = df[df["image_path"].notna()].copy()
lab = np.stack(df["Finding Labels"].apply(encode).to_numpy())
for i,c in enumerate(CLASSES): df[c]=lab[:,i]

# patient-wise 70/15/15
groups = df["Patient ID"].values
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss1.split(df, groups=groups))
df_tv, df_te = df.iloc[train_val_idx].copy(), df.iloc[test_idx].copy()
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss2.split(df_tv, groups=df_tv["Patient ID"].values))
df_tr, df_va = df_tv.iloc[train_idx].copy(), df_tv.iloc[val_idx].copy()

df_tr[["image_path","Patient ID"]+CLASSES].to_csv("/kaggle/working/train.csv",index=False)
df_va[["image_path","Patient ID"]+CLASSES].to_csv("/kaggle/working/val.csv",index=False)
df_te[["image_path","Patient ID"]+CLASSES].to_csv("/kaggle/working/test.csv",index=False)

# pos_weight
pos = df_tr[CLASSES].sum(0).values
neg = len(df_tr)-pos
pw = (neg/np.clip(pos,1,None)).astype(float)
with open("/kaggle/working/pos_weight.json","w") as f:
    json.dump({c:float(w) for c,w in zip(CLASSES,pw)}, f, indent=2)

print("Splits built:")
print(" Train:",len(df_tr)," Val:",len(df_va)," Test:",len(df_te))
print("pos_weight.json saved")


Using metadata: /kaggle/input/data/Data_Entry_2017.csv
Splits built:
 Train: 78298  Val: 17098  Test: 16724
pos_weight.json saved


# Step D2-fix — remove 1000-class head from the checkpoint, then load backbone

In [2]:
import torch, timm

# --- Step 1: Define 14 disease classes ---
CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule',
    'Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema',
    'Fibrosis','Pleural_Thickening','Hernia'
]

# --- Step 2: Optional pretrained DenseNet checkpoint (if you trained one yourself) ---
# Example: "/kaggle/input/my-densenet121-weights/densenet121_14cls.pth"
# If you don't have one, leave this as None
WEIGHTS_PATH = None

# --- Step 3: Build DenseNet121 model for 14-class classification ---
# Using pretrained=True loads ImageNet weights (recommended for transfer learning)
model = timm.create_model("densenet121", pretrained=False, num_classes=len(CLASSES))
print("✅ DenseNet121 model created with ImageNet pretrained weights.")
print("Output head adjusted for", len(CLASSES), "classes.")

# --- Step 4: If you have your own DenseNet checkpoint, load it here ---
if WEIGHTS_PATH is not None:
    state = torch.load(WEIGHTS_PATH, map_location="cpu")
    if "state_dict" in state:
        state = state["state_dict"]

    # Clean prefix like "module." if present
    clean_state = {}
    for k, v in state.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[7:]
        clean_state[nk] = v

    msg = model.load_state_dict(clean_state, strict=False)
    print("✅ Loaded custom DenseNet121 checkpoint:", msg)

# --- Step 5: Sanity check forward pass ---
model.eval()
with torch.no_grad():
    y = model(torch.randn(2, 3, 224, 224))
print("Output shape:", tuple(y.shape))  # should be (2, 14)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ DenseNet121 model created with ImageNet pretrained weights.
Output head adjusted for 14 classes.
Output shape: (2, 14)


# Step D3 — DenseNet121 + DataLoaders + one-batch sanity (no training yet)

In [3]:

import os, json, cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A

# Disable Albumentations version check
os.environ["ALBUMENTATIONS_DISABLE_VERSION_CHECK"] = "1"

# ---------------------- Configuration ----------------------
TRAIN_CSV = "/kaggle/working/train.csv"
VAL_CSV   = "/kaggle/working/val.csv"
TEST_CSV  = "/kaggle/working/test.csv"

CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule',
    'Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema',
    'Fibrosis','Pleural_Thickening','Hernia'
]
IMG_SIZE, BATCH_SIZE = 224, 16
device = torch.device("cpu")  # change to "cuda" if GPU is available

# ---------------------- Image Transformations ----------------------
# Conservative augmentations for CNN-based models (DenseNet121)
train_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
    A.HorizontalFlip(p=0.5),
    A.Affine(rotate=(-8,8), translate_percent={"x":0.02,"y":0.02}, scale=(0.9,1.1), p=0.3),
    A.RandomBrightnessContrast(p=0.15),
    A.Normalize(mean=(0.485,), std=(0.229,)),
])
val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
    A.Normalize(mean=(0.485,), std=(0.229,)),
])

# ---------------------- Dataset Class ----------------------
class CXRDataset(Dataset):
    def __init__(self, csv_path, is_train=True):
        self.df = pd.read_csv(csv_path)
        self.tfms = train_tfms if is_train else val_tfms
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(r["image_path"])
        img = img[..., None]
        img = self.tfms(image=img)["image"]
        x = torch.tensor(np.transpose(np.repeat(img, 3, axis=2), (2,0,1)), dtype=torch.float32)
        y = torch.tensor(r[CLASSES].values.astype(np.float32))
        return x, y

# ---------------------- Dataloaders ----------------------
train_loader = DataLoader(CXRDataset(TRAIN_CSV, True), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(CXRDataset(VAL_CSV,   False), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

# ---------------------- Loss Function ----------------------
with open("/kaggle/working/pos_weight.json","r") as f:
    pw = json.load(f)
pos_weight = torch.tensor([pw[c] for c in CLASSES], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ---------------------- Sanity Check ----------------------
xb, yb = next(iter(train_loader))
with torch.no_grad():
    logits = model(xb.to(device))
    loss = criterion(logits, yb.to(device))
print(f"Batch input shape: {tuple(xb.shape)}  |  Label shape: {tuple(yb.shape)}")
print(f"Model output shape: {tuple(logits.shape)}")
print(f"Sanity BCE loss: {float(loss):.4f}")
#completed

Batch input shape: (16, 3, 224, 224)  |  Label shape: (16, 14)
Model output shape: (16, 14)
Sanity BCE loss: 1.0962


# Step D4 — Training (head-only → finetune), early stop on val micro-AUC


In [7]:
# Step D4 — Training (head-only → finetune), early stop on val micro-AUC
import time, numpy as np, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, f1_score

# -----------------------------
# Device setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

CKPT_PATH = "/kaggle/working/densenet121_best.pt"

# --- Training hyperparameters ---
HEAD_EPOCHS   = 2       # Train classifier layer only first
FT_EPOCHS     = 6       # Then fine-tune full model
PATIENCE      = 2
MAX_TRAIN_STEPS = 300
LR_HEAD, WD_HEAD = 1e-3, 1e-4
LR_FT,   WD_FT   = 1e-4, 5e-2

# ✅ Move model & criterion to GPU
model = model.to(device)
criterion = criterion.to(device)

# -------------------------
# Evaluation function
# -------------------------
def evaluate(model, loader, thr=0.5):
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            P.append(probs)
            Y.append(yb.cpu().numpy())
    P, Y = np.vstack(P), np.vstack(Y)
    micro_auc = roc_auc_score(Y, P, average='micro')
    preds = (P >= thr).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    return float(micro_auc), float(micro_f1)

# -------------------------
# Train one epoch
# -------------------------
def train_one_epoch(loader, optimizer, step_cap=None):
    model.train()
    total, n, steps = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
        n += xb.size(0)
        steps += 1
        if step_cap and steps >= step_cap:
            break
    return total / max(1, n)

# -------------------------
# Full training phase
# -------------------------
def run_phase(name, epochs, optimizer):
    best_auc, bad = -1.0, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(train_loader, optimizer, step_cap=MAX_TRAIN_STEPS)
        val_auc, val_f1 = evaluate(model, val_loader, thr=0.5)
        mins = (time.time() - t0) / 60
        print(f"[{name}] Ep{ep:02d} | loss={tr_loss:.4f} | val micro-AUC={val_auc:.4f} | micro-F1={val_f1:.4f} | {mins:.1f}m")
        
        if val_auc > best_auc:
            best_auc, bad = val_auc, 0
            torch.save({"state_dict": model.state_dict()}, CKPT_PATH)
            print("  ✅ New best — saved to", CKPT_PATH)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  ⏹ Early stop (no AUC improve {PATIENCE} epochs). Best={best_auc:.4f}")
                break
    return best_auc

# ---- Phase 1: freeze backbone, train classifier only ----
for n, p in model.named_parameters():
    p.requires_grad = n.startswith("classifier")

opt_head = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=LR_HEAD, weight_decay=WD_HEAD)
print("=== Phase 1: Classifier-only training ===")
best1 = run_phase("head", HEAD_EPOCHS, opt_head)

# ---- Phase 2: unfreeze all layers, fine-tune entire DenseNet ----
for p in model.parameters():
    p.requires_grad = True

opt_ft = torch.optim.AdamW(model.parameters(), lr=LR_FT, weight_decay=WD_FT)
print("=== Phase 2: Fine-tuning entire model ===")
best2 = run_phase("finetune", FT_EPOCHS, opt_ft)

print("✅ Training complete. Best validation micro-AUC observed:", round(max(best1, best2), 4))


✅ Using device: cuda
=== Phase 1: Classifier-only training ===
[head] Ep01 | loss=1.5260 | val micro-AUC=0.5936 | micro-F1=0.1229 | 9.6m
  ✅ New best — saved to /kaggle/working/densenet121_best.pt
[head] Ep02 | loss=1.3733 | val micro-AUC=0.5112 | micro-F1=0.0994 | 7.8m
=== Phase 2: Fine-tuning entire model ===
[finetune] Ep01 | loss=1.3314 | val micro-AUC=0.5763 | micro-F1=0.1104 | 8.3m
  ✅ New best — saved to /kaggle/working/densenet121_best.pt
[finetune] Ep02 | loss=1.2720 | val micro-AUC=0.6275 | micro-F1=0.1220 | 8.0m
  ✅ New best — saved to /kaggle/working/densenet121_best.pt


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


[finetune] Ep03 | loss=1.3062 | val micro-AUC=0.6174 | micro-F1=0.1266 | 8.1m


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


[finetune] Ep04 | loss=1.2853 | val micro-AUC=0.6576 | micro-F1=0.1426 | 8.3m
  ✅ New best — saved to /kaggle/working/densenet121_best.pt


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


[finetune] Ep05 | loss=1.2895 | val micro-AUC=0.6005 | micro-F1=0.1152 | 8.1m


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


[finetune] Ep06 | loss=1.2524 | val micro-AUC=0.6671 | micro-F1=0.1359 | 8.5m
  ✅ New best — saved to /kaggle/working/densenet121_best.pt
✅ Training complete. Best validation micro-AUC observed: 0.6671


# Step D5 — Threshold tuning (val) + Final test evaluation


In [9]:
# ==============================
# Step D5 — Threshold tuning (val) + Final test evaluation
# ==============================
import json, numpy as np, torch
from sklearn.metrics import roc_auc_score, f1_score
from torch.utils.data import DataLoader

# --- Paths & device ---
CKPT_PATH = "/kaggle/working/densenet121_best.pt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# --- Recreate test_loader (important fix) ---
# Reuse your dataset class and transforms used for val_loader
# Make sure CXRDataset and TEST_CSV are already defined
TEST_BATCH = BATCH_SIZE * 2 if "BATCH_SIZE" in locals() else 32
NUM_WORKERS = 2

test_loader = DataLoader(
    CXRDataset(TEST_CSV, is_train=False),
    batch_size=TEST_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("✅ test_loader ready | Batch size:", TEST_BATCH, "| Workers:", NUM_WORKERS)

# --- Load best DenseNet121 checkpoint ---
sd = torch.load(CKPT_PATH, map_location=device)["state_dict"]
model.load_state_dict(sd, strict=False)
model = model.to(device)
model.eval()
print(f"✅ Loaded checkpoint from: {CKPT_PATH}")

# ------------------------------
# Prediction helper
# ------------------------------
def predict(loader, tta=False):
    Ps, Ys = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            if tta:
                logits1 = model(xb).cpu().numpy()
                logits2 = model(torch.flip(xb, dims=[3])).cpu().numpy()
                probs = (1/(1+np.exp(-logits1)) + 1/(1+np.exp(-logits2))) / 2
            else:
                logits = model(xb).cpu().numpy()
                probs = 1/(1+np.exp(-logits))
            Ps.append(probs)
            Ys.append(yb.numpy())
    return np.vstack(Ps), np.vstack(Ys)

# ------------------------------
# Metric computation
# ------------------------------
def metrics(P, Y, thr):
    micro_auc = roc_auc_score(Y, P, average='micro')

    aucs = []
    for j in range(P.shape[1]):
        if Y[:, j].sum() > 0:
            try:
                aucs.append(roc_auc_score(Y[:, j], P[:, j]))
            except:
                pass
    macro_auc = float(np.mean(aucs)) if aucs else float("nan")

    preds = (P >= thr).astype(int) if isinstance(thr, float) else (P >= np.asarray(thr)[None, :]).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(Y, preds, average='macro', zero_division=0)

    return dict(
        microAUC=float(micro_auc),
        macroAUC=float(macro_auc),
        microF1=float(micro_f1),
        macroF1=float(macro_f1)
    )

# ------------------------------
# 1️⃣ Tune thresholds on Validation
# ------------------------------
print("Predicting on validation set (no TTA)...")
P_val, Y_val = predict(val_loader, tta=False)

grid = np.linspace(0.05, 0.95, 19)
thr_vec = np.zeros(P_val.shape[1], dtype=np.float32)

for j in range(P_val.shape[1]):
    if Y_val[:, j].sum() == 0:
        thr_vec[j] = 0.5
        continue
    f1s = [f1_score(Y_val[:, j], (P_val[:, j] >= t).astype(int), zero_division=0) for t in grid]
    thr_vec[j] = float(grid[int(np.argmax(f1s))])

val_base  = metrics(P_val, Y_val, thr=0.5)
val_tuned = metrics(P_val, Y_val, thr=thr_vec)

print(f"[VAL] base  microAUC={val_base['microAUC']:.4f} macroAUC={val_base['macroAUC']:.4f} microF1={val_base['microF1']:.4f} macroF1={val_base['macroF1']:.4f}")
print(f"[VAL] tuned microAUC={val_tuned['microAUC']:.4f} macroAUC={val_tuned['macroAUC']:.4f} microF1={val_tuned['microF1']:.4f} macroF1={val_tuned['macroF1']:.4f}")

# Save thresholds for reproducibility
with open("/kaggle/working/thresholds.json", "w") as f:
    json.dump({c: float(t) for c, t in zip(CLASSES, thr_vec)}, f, indent=2)
print("✅ Saved thresholds.json")

# ------------------------------
# 2️⃣ Final Test evaluation
# ------------------------------
print("Predicting on test set (no TTA)...")
P_test, Y_test = predict(test_loader, tta=False)
test_tuned = metrics(P_test, Y_test, thr=thr_vec)
print(f"[TEST] tuned microAUC={test_tuned['microAUC']:.4f} macroAUC={test_tuned['macroAUC']:.4f} microF1={test_tuned['microF1']:.4f} macroF1={test_tuned['macroF1']:.4f}")

# ------------------------------
# 3️⃣ Test with TTA (flip augmentation)
# ------------------------------
print("Predicting on test set (with TTA)...")
P_test_tta, _ = predict(test_loader, tta=True)
test_tuned_tta = metrics(P_test_tta, Y_test, thr=thr_vec)
print(f"[TEST+TTA] tuned microAUC={test_tuned_tta['microAUC']:.4f} macroAUC={test_tuned_tta['macroAUC']:.4f} microF1={test_tuned_tta['microF1']:.4f} macroF1={test_tuned_tta['macroF1']:.4f}")

# ------------------------------
# 4️⃣ Save all final metrics
# ------------------------------
with open("/kaggle/working/metrics_final.json", "w") as f:
    json.dump({
        "val_base": val_base,
        "val_tuned": val_tuned,
        "test_tuned": test_tuned,
        "test_tuned_tta": test_tuned_tta,
        "thresholds": {c: float(t) for c, t in zip(CLASSES, thr_vec)}
    }, f, indent=2)

print("✅ Saved metrics_final.json & thresholds.json in /kaggle/working")
print("🎯 Step D5 complete — DenseNet121 evaluation finished successfully.")


✅ Using device: cuda
✅ test_loader ready | Batch size: 32 | Workers: 2
✅ Loaded checkpoint from: /kaggle/working/densenet121_best.pt
Predicting on validation set (no TTA)...
[VAL] base  microAUC=0.6671 macroAUC=0.6633 microF1=0.1359 macroF1=0.1247
[VAL] tuned microAUC=0.6671 macroAUC=0.6633 microF1=0.1920 macroF1=0.1401
✅ Saved thresholds.json
Predicting on test set (no TTA)...
[TEST] tuned microAUC=0.6633 macroAUC=0.6539 microF1=0.1858 macroF1=0.1367
Predicting on test set (with TTA)...
[TEST+TTA] tuned microAUC=0.6634 macroAUC=0.6550 microF1=0.1853 macroF1=0.1367
✅ Saved metrics_final.json & thresholds.json in /kaggle/working
🎯 Step D5 complete — DenseNet121 evaluation finished successfully.


# Low-LR polish: resume from best, 2-3 short epochs at LR=5e-5 (DenseNet121)


In [10]:
# ------------------------------
# Low-LR polish: resume from best, 2-3 short epochs at LR=5e-5 (DenseNet121, GPU-ready)
# ------------------------------
import torch, time, numpy as np
from sklearn.metrics import roc_auc_score, f1_score

CKPT_PATH = "/kaggle/working/densenet121_best.pt"   # <-- ensure this is your DenseNet checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", device)

# ------------------------------
# Reload best weights into model (clean possible 'module.' prefix)
# ------------------------------
sd = torch.load(CKPT_PATH, map_location="cpu")
# unwrap if saved as {"state_dict": ...}
if isinstance(sd, dict) and "state_dict" in sd:
    sd = sd["state_dict"]

# clean prefixes (common when saving DataParallel models)
clean_sd = {}
for k, v in sd.items():
    nk = k
    if nk.startswith("module."):
        nk = nk[7:]
    clean_sd[nk] = v

model.load_state_dict(clean_sd, strict=False)
model = model.to(device)
print("✅ Loaded checkpoint from:", CKPT_PATH)

# ------------------------------
# Ensure criterion is on device
# ------------------------------
criterion = criterion.to(device)

# ------------------------------
# Make sure all params are trainable for polish
# ------------------------------
for p in model.parameters():
    p.requires_grad = True

# ------------------------------
# Optimizer & polish hyperparams
# ------------------------------
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=5e-5, weight_decay=5e-2)
PATIENCE = 2
MAX_TRAIN_STEPS = 300   # keep small for short polish epochs; set None for full epoch if desired
POLISH_EPOCHS = 3       # 2-3 short epochs recommended

# ------------------------------
# Evaluation helper (device aware)
# ------------------------------
def evaluate(loader):
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            probs  = torch.sigmoid(logits).detach().cpu().numpy()
            P.append(probs)
            Y.append(yb.detach().cpu().numpy())
    P = np.vstack(P)
    Y = np.vstack(Y)
    micro_auc = roc_auc_score(Y, P, average='micro')
    preds = (P >= 0.5).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    return float(micro_auc), float(micro_f1)

# ------------------------------
# Train one epoch (device aware)
# ------------------------------
def train_one_epoch():
    model.train()
    tot, n, steps = 0.0, 0, 0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        tot += loss.item() * xb.size(0)
        n += xb.size(0)
        steps += 1
        if MAX_TRAIN_STEPS and steps >= MAX_TRAIN_STEPS:
            break
    return tot / max(1, n)

# ------------------------------
# Polish training loop with early stopping on val micro-AUC
# ------------------------------
best_auc, bad = -1.0, 0
for ep in range(1, POLISH_EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_one_epoch()
    val_auc, val_f1 = evaluate(val_loader)
    elapsed = (time.time() - t0) / 60.0
    print(f"[polish] Ep{ep:02d} | loss={tr_loss:.4f} | val micro-AUC={val_auc:.4f} | micro-F1={val_f1:.4f} | {elapsed:.2f}m")
    if val_auc > best_auc:
        best_auc, bad = val_auc, 0
        torch.save({"state_dict": model.state_dict()}, CKPT_PATH)
        print("  ✅ New best saved:", CKPT_PATH)
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"  ⏹ Early stop (no AUC improve {PATIENCE} epochs). Best={best_auc:.4f}")
            break

print("✅ Polish pass done. Best validation micro-AUC observed:", round(best_auc, 4))


✅ Device: cuda
✅ Loaded checkpoint from: /kaggle/working/densenet121_best.pt
[polish] Ep01 | loss=1.2223 | val micro-AUC=0.6711 | micro-F1=0.1401 | 8.40m
  ✅ New best saved: /kaggle/working/densenet121_best.pt
[polish] Ep02 | loss=1.2148 | val micro-AUC=0.6984 | micro-F1=0.1434 | 8.31m
  ✅ New best saved: /kaggle/working/densenet121_best.pt
[polish] Ep03 | loss=1.2323 | val micro-AUC=0.6885 | micro-F1=0.1504 | 8.25m
✅ Polish pass done. Best validation micro-AUC observed: 0.6984


# Step D5 (refresh) — tune thresholds on val, then evaluate test with polished checkpoint


In [11]:
# ============================
# Step D6 (refresh) — tune thresholds on val, then evaluate test with polished checkpoint
# DenseNet121 GPU-ready, strict version
# ============================
import numpy as np
import torch
import json
from sklearn.metrics import roc_auc_score, f1_score

# ----- Config -----
CKPT_PATH = "/kaggle/working/densenet121_best.pt"   # <-- ensure this points to your DenseNet checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Device:", device)

# ----- Reload best weights after polish (robustly) -----
sd = torch.load(CKPT_PATH, map_location="cpu")
if isinstance(sd, dict) and "state_dict" in sd:
    sd = sd["state_dict"]

# clean 'module.' if present
clean_sd = {}
for k, v in sd.items():
    nk = k
    if nk.startswith("module."):
        nk = nk[7:]
    clean_sd[nk] = v

model.load_state_dict(clean_sd, strict=False)
model = model.to(device)
model.eval()
print(f"✅ Loaded checkpoint from: {CKPT_PATH}")

# ----- Prediction (device-aware) -----
def predict(loader, tta=False):
    Ps, Ys = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)         # move batch to device
            yb = yb.to(device)
            if tta:
                # base prediction
                logits1 = model(xb)
                # horizontal flip (width dim = 3)
                logits2 = model(torch.flip(xb, dims=[3]))
                probs = (torch.sigmoid(logits1) + torch.sigmoid(logits2)) / 2.0
            else:
                logits = model(xb)
                probs = torch.sigmoid(logits)
            Ps.append(probs.detach().cpu().numpy())
            Ys.append(yb.detach().cpu().numpy())
    P = np.vstack(Ps)
    Y = np.vstack(Ys)
    return P, Y

# ----- Metrics (supports scalar or per-class thr vector) -----
def metrics(P, Y, thr):
    micro_auc = roc_auc_score(Y, P, average='micro')
    aucs = []
    for j in range(P.shape[1]):
        if Y[:, j].sum() > 0:
            try:
                aucs.append(roc_auc_score(Y[:, j], P[:, j]))
            except Exception:
                pass
    macro_auc = float(np.mean(aucs)) if aucs else float("nan")

    if isinstance(thr, float):
        preds = (P >= thr).astype(int)
    else:
        thr_arr = np.asarray(thr, dtype=float)
        preds = (P >= thr_arr[None, :]).astype(int)

    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(Y, preds, average='macro', zero_division=0)
    return {"microAUC": float(micro_auc), "macroAUC": float(macro_auc),
            "microF1": float(micro_f1), "macroF1": float(macro_f1)}

# ----- 1) Tune thresholds on validation -----
print("Predicting on validation set (no TTA)...")
P_val, Y_val = predict(val_loader, tta=False)

grid = np.linspace(0.05, 0.95, 19)
thr_vec = np.zeros(P_val.shape[1], dtype=np.float32)
for j in range(P_val.shape[1]):
    if Y_val[:, j].sum() == 0:
        thr_vec[j] = 0.5
        continue
    f1s = [f1_score(Y_val[:, j], (P_val[:, j] >= t).astype(int), zero_division=0) for t in grid]
    thr_vec[j] = float(grid[int(np.argmax(f1s))])

val_tuned = metrics(P_val, Y_val, thr=thr_vec)
print(f"[VAL-polished] microAUC={val_tuned['microAUC']:.4f} macroAUC={val_tuned['macroAUC']:.4f} "
      f"microF1={val_tuned['microF1']:.4f} macroF1={val_tuned['macroF1']:.4f}")

# ----- 2) Test eval (no TTA) -----
print("Predicting on test set (no TTA)...")
P_test, Y_test = predict(test_loader, tta=False)
test_tuned = metrics(P_test, Y_test, thr=thr_vec)
print(f"[TEST-polished] microAUC={test_tuned['microAUC']:.4f} macroAUC={test_tuned['macroAUC']:.4f} "
      f"microF1={test_tuned['microF1']:.4f} macroF1={test_tuned['macroF1']:.4f}")

# ----- 3) Test + TTA eval ----- 
print("Predicting on test set (with TTA)...")
P_test_tta, _ = predict(test_loader, tta=True)
test_tuned_tta = metrics(P_test_tta, Y_test, thr=thr_vec)
print(f"[TEST+TTA-polished] microAUC={test_tuned_tta['microAUC']:.4f} macroAUC={test_tuned_tta['macroAUC']:.4f} "
      f"microF1={test_tuned_tta['microF1']:.4f} macroF1={test_tuned_tta['macroF1']:.4f}")

# ----- 4) Save results -----
out = {
    "val_tuned": val_tuned,
    "test_tuned": test_tuned,
    "test_tuned_tta": test_tuned_tta,
    "thresholds": {c: float(t) for c, t in zip(CLASSES, thr_vec)}
}
with open("/kaggle/working/metrics_final_polished.json", "w") as f:
    json.dump(out, f, indent=2)

print("✅ Saved: /kaggle/working/metrics_final_polished.json")


✅ Device: cuda
✅ Loaded checkpoint from: /kaggle/working/densenet121_best.pt
Predicting on validation set (no TTA)...
[VAL-polished] microAUC=0.6984 macroAUC=0.6757 microF1=0.1969 macroF1=0.1468
Predicting on test set (no TTA)...
[TEST-polished] microAUC=0.6905 macroAUC=0.6724 microF1=0.1893 macroF1=0.1397
Predicting on test set (with TTA)...
[TEST+TTA-polished] microAUC=0.6921 macroAUC=0.6748 microF1=0.1906 macroF1=0.1408
✅ Saved: /kaggle/working/metrics_final_polished.json


# Finalize: save polished DenseNet121 checkpoint + package outputs

In [12]:
# -----------------------------
# Finalize: save polished DenseNet121 checkpoint + package outputs
# -----------------------------
import torch, shutil, json, zipfile, os

# ----- Config: point to your DenseNet checkpoint names -----
FINAL_MODEL_PATH = "/kaggle/working/densenet121_final.pt"
BEST_SD_PATH     = "/kaggle/working/densenet121_best.pt"    # from training/polish
WORKDIR = "/kaggle/working"

# ----- 1) Load best state dict robustly -----
if not os.path.exists(BEST_SD_PATH):
    raise FileNotFoundError(f"Best checkpoint not found: {BEST_SD_PATH}")

sd_loaded = torch.load(BEST_SD_PATH, map_location="cpu")
# If a wrapper dict was saved, unwrap
if isinstance(sd_loaded, dict) and "state_dict" in sd_loaded:
    best_sd = sd_loaded["state_dict"]
else:
    best_sd = sd_loaded

# Optional: strip "module." prefix if present (from DataParallel)
clean_sd = {}
for k, v in best_sd.items():
    nk = k
    if nk.startswith("module."):
        nk = nk[7:]
    clean_sd[nk] = v

# ----- 2) Save final checkpoint (state_dict + classes) -----
# Ensure CLASSES exists in the notebook environment
if "CLASSES" not in globals():
    raise RuntimeError("CLASSES variable not found in environment. Define CLASSES before running this cell.")

torch.save({"state_dict": clean_sd, "classes": CLASSES}, FINAL_MODEL_PATH)
print("✅ Final model saved to:", FINAL_MODEL_PATH)

# ----- 3) Collect files to package (must match actual produced files) -----
files_to_keep = [
    os.path.basename(FINAL_MODEL_PATH),
    "train.csv", "val.csv", "test.csv",
    "pos_weight.json",
    "metrics_final_polished.json",
    "thresholds.json"
]

# Verify each file exists under WORKDIR
missing = [f for f in files_to_keep if not os.path.exists(os.path.join(WORKDIR, f))]
if missing:
    raise FileNotFoundError("Missing files in workdir: " + ", ".join(missing))

# ----- 4) Write project manifest -----
manifest = {
    "model_file": os.path.basename(FINAL_MODEL_PATH),
    "splits": ["train.csv", "val.csv", "test.csv"],
    "pos_weight": "pos_weight.json",
    "metrics": "metrics_final_polished.json",
    "thresholds": "thresholds.json",
    "classes": CLASSES
}
manifest_path = os.path.join(WORKDIR, "project_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print("✅ Manifest written to:", manifest_path)

# ----- 5) Zip everything into a single package -----
zip_path = os.path.join(WORKDIR, "densenet_package.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for fname in files_to_keep + ["project_manifest.json"]:
        src = os.path.join(WORKDIR, fname)
        z.write(src, arcname=fname)
print("✅ Created archive:", zip_path)


✅ Final model saved to: /kaggle/working/densenet121_final.pt
✅ Manifest written to: /kaggle/working/project_manifest.json
✅ Created archive: /kaggle/working/densenet_package.zip


# Step D7 — Classic metrics (Accuracy, Precision, Recall, F1)

In [13]:
# ==============================================
# Step D7 — Classic metrics (Accuracy, Precision, Recall, F1)
# for DenseNet121 model on GPU
# ==============================================
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Configuration ---
CKPT_PATH = "/kaggle/working/densenet121_best.pt"   # Use your DenseNet checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load best model weights ---
print("Using device:", device)
sd = torch.load(CKPT_PATH, map_location=device)
state_dict = sd["state_dict"] if "state_dict" in sd else sd
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

# --- Prediction helper (same style as earlier) ---
def predict(loader, tta=False):
    Ps, Ys = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            if tta:
                logits1 = model(xb).cpu().numpy()
                logits2 = model(torch.flip(xb, dims=[3])).cpu().numpy()
                probs = (1 / (1 + np.exp(-logits1)) + 1 / (1 + np.exp(-logits2))) / 2
            else:
                logits = model(xb).cpu().numpy()
                probs = 1 / (1 + np.exp(-logits))
            Ps.append(probs)
            Ys.append(yb.numpy())
    return np.vstack(Ps), np.vstack(Ys)

# --- Evaluation function for classic metrics ---
def eval_classic(P, Y, thr=0.5):
    preds = (P >= thr).astype(int)
    
    # Micro-level metrics
    acc_micro = accuracy_score(Y.flatten(), preds.flatten())
    prec_micro = precision_score(Y.flatten(), preds.flatten(), average='micro', zero_division=0)
    rec_micro = recall_score(Y.flatten(), preds.flatten(), average='micro', zero_division=0)
    f1_micro = f1_score(Y.flatten(), preds.flatten(), average='micro', zero_division=0)
    
    # Macro-level metrics
    prec_macro = precision_score(Y, preds, average='macro', zero_division=0)
    rec_macro = recall_score(Y, preds, average='macro', zero_division=0)
    f1_macro = f1_score(Y, preds, average='macro', zero_division=0)
    
    return {
        "Accuracy": acc_micro,
        "Precision - Micro": prec_micro,
        "Recall - Micro": rec_micro,
        "F1-Score - Micro": f1_micro,
        "Precision - Macro": prec_macro,
        "Recall - Macro": rec_macro,
        "F1-Score - Macro": f1_macro,
    }

# --- Run evaluation on Test Set ---
print("\nPredicting on test set with tuned thresholds...")
P_test, Y_test = predict(test_loader, tta=False)

# Ensure thr_vec exists (from your tuned D5/D6 step)
if 'thr_vec' not in globals():
    raise RuntimeError("thr_vec not found — run D5/D6 threshold tuning first.")

classic_metrics = eval_classic(P_test, Y_test, thr=thr_vec)

print("\n=== Classic Metrics on Test (DenseNet121 + Tuned Thresholds) ===")
for k, v in classic_metrics.items():
    print(f"{k}: {v:.4f}")

# --- Save metrics to disk ---
import json
with open("/kaggle/working/classic_metrics_test.json", "w") as f:
    json.dump(classic_metrics, f, indent=2)

print("✅ Saved classic_metrics_test.json")


Using device: cuda

Predicting on test set with tuned thresholds...

=== Classic Metrics on Test (DenseNet121 + Tuned Thresholds) ===
Accuracy: 0.8160
Precision - Micro: 0.8160
Recall - Micro: 0.8160
F1-Score - Micro: 0.8160
Precision - Macro: 0.0971
Recall - Macro: 0.3230
F1-Score - Macro: 0.1397
✅ Saved classic_metrics_test.json


# Densenet121 done!